[EN]

# 03 — Analysis

Analyze the trajectory of `02_run_md.ipynb`: charge transfer, decomposition events, reaction products,
step-by-step decomposition of selected molecules, and the elemental composition near the Li surface.

[JA]

# 03 — 解析

`02_run_md.ipynb` の軌跡について、電荷移動、分解イベント、反応生成物、分子の分解過程、Li 表面近くの元素組成を解析します。

[EN]

## Imports

[JA]

## ライブラリのインポート

In [ ]:
%load_ext watermark
%watermark -v -p numpy,pandas,matplotlib,scipy,ase,pfcc_extras

In [ ]:
import json
import os
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ase.data import atomic_masses, atomic_numbers
from ase.io import read
from pfcc_extras.visualize.view import view_ngl

from src.analysis import (
    bond_analysis,
    distance_from_surface,
    first_persistent,
    fragment_formula,
    hill,
    region_counts,
    to_percent,
    whole_molecule,
)

[EN]

## Parameters

`run_name` selects the run of `02_run_md.ipynb` to analyze (`"md_nvt"`, or `"md_quick_test"` for the quick test).

[JA]

## パラメータ

`run_name` で、解析する `02_run_md.ipynb` の計算を選びます（本番は `"md_nvt"`、動作確認は `"md_quick_test"`）。

In [ ]:
dir_path = Path.cwd()
interface_dir = dir_path / "data" / "01_build_interface"  # from 01_build_interface.ipynb
md_dir = dir_path / "data" / "02_run_md"                  # from 02_run_md.ipynb
output_dir = dir_path / "data" / "03_analysis"

run_name = "md_nvt"           # "md_nvt" or "md_quick_test"
frame_stride = 1              # analyze every n-th frame
bond_scale = 1.2              # bonded if distance < bond_scale x (sum of covalent radii)
persistence = 3               # frames a molecule must stay split to count as decomposed
closeup_molecules = None      # e.g. ["PF6_003", "EMC_038"]; None = earliest decomposed molecule of each species

sei_cutoff_A = 5.0            # SEI region: within this distance of the outermost Li layer (A)
composition_window_ps = 5.0   # average over the last part of the run (ps)
elements = ["C", "O", "F", "P"]
sensitivity_cutoffs = [3.0, 4.0, 5.0, 6.0, 8.0]
profile_range_A = (-6.0, 12.0)
profile_bin_A = 0.25

# Optional reference composition, e.g. {"label": "XPS", "basis": "atomic", "values": {"C": 40, "O": 40, "F": 15, "P": 5}}
# basis: "atomic" (at%) or "weight" (wt%)
reference_composition = None

charge_plot_file = output_dir / "charge_slab_electrolyte.png"
charge_molecule_plot_file = output_dir / "charge_per_molecule.png"
charge_csv_file = output_dir / "charge_per_molecule.csv"
events_csv_file = output_dir / "decomposition_events.csv"
events_plot_file = output_dir / "decomposition_events.png"
products_csv_file = output_dir / "products.csv"
products_plot_file = output_dir / "products_time.png"
closeup_plot_file = output_dir / "closeup_{label}.png"
composition_csv_file = output_dir / "sei_composition.csv"
profile_plot_file = output_dir / "sei_density_profile.png"
composition_plot_file = output_dir / "sei_composition.png"

os.makedirs(output_dir, exist_ok=True)

In [ ]:
system = json.loads((interface_dir / "system_info.json").read_text())
run = json.loads((md_dir / f"{run_name}_info.json").read_text())
frames = read(md_dir / run["trajectory"], index=f"::{frame_stride}")
times = np.arange(len(frames)) * run["save_interval_fs"] * frame_stride / 1000  # ps

symbols = np.array(frames[0].get_chemical_symbols())
n_atoms = len(symbols)
n_slab = len(system["slab_indices"])
slab_idx = np.arange(n_slab)
elec_idx = np.arange(n_slab, n_atoms)

mols = pd.DataFrame(system["molecules"])
species_list = list(dict.fromkeys(mols["species"]))
mol_of_atom = np.full(n_atoms, -1)
for k, (start, end) in enumerate(zip(mols["start"], mols["end"])):
    mol_of_atom[start:end] = k

positions = np.array([f.positions for f in frames])
cell = frames[0].cell.array
area = cell[0, 0] * cell[1, 1]
print(f"{run_name}: {len(frames)} frames, 0-{times[-1]:.1f} ps, {n_atoms} atoms")

[EN]

## Distance from the Li surface

`dist` is the distance of every atom from the nearer outermost Li layer: positive in the electrolyte, negative inside
the slab. The fixed central layers are used as the reference, so no re-centering is needed.

[JA]

## Li 表面からの距離

`dist` は、各原子の、近い方の最外層の Li からの距離です。電解液側が正、スラブの内側が負です。固定した中央の層を基準にするので、軌跡の中心を合わせ直す必要はありません。

In [ ]:
dist, top_plane, bottom_plane = distance_from_surface(
    positions, cell, system["fixed_indices"], system["top_layer_indices"], system["bottom_layer_indices"]
)

[EN]

## Charge transfer

The atomic charges are summed over the slab, the electrolyte and each molecule. The total stays zero, so the charge lost
by the slab is gained by the electrolyte. Read the charges as an indicator of electron transfer.

[JA]

## 電荷移動

原子電荷を、スラブ・電解液・分子ごとに合計します。系全体は 0 なので、スラブが失った電荷は電解液が受け取った電荷です。電子移動の目安として読んでください。

In [ ]:
try:
    charges = np.array([f.get_charges() for f in frames])
except Exception:
    charges = None
    print("No atomic charges in the trajectory: the charge analysis is skipped.")

if charges is not None:
    q_slab = charges[:, slab_idx].sum(axis=1)
    q_elec = charges[:, elec_idx].sum(axis=1)
    q_mol = np.stack([charges[:, s:e].sum(axis=1) for s, e in zip(mols["start"], mols["end"])], axis=1)

    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.plot(times, q_slab, label="Li slab")
    ax.plot(times, q_elec, label="Electrolyte")
    ax.plot(times, q_slab + q_elec, color="gray", ls=":", label="Total")
    ax.set_xlabel("Time (ps)")
    ax.set_ylabel("Charge (e)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(charge_plot_file, dpi=150)
    plt.show()
    print(f"Charge of the slab: {q_slab[0]:+.1f} e -> {q_slab[-1]:+.1f} e")

[EN]

## Decomposition events

In every frame, bonds within each original molecule are detected from interatomic distances (Li excluded), and the
molecule is split into connected fragments (`bond_analysis`). A molecule counts as decomposed from the first frame at
which it stays split for `persistence` frames.

[JA]

## 分解イベント

各フレームで、元の分子の中の結合を原子間距離から判定し（Li は除く）、分子がいくつのフラグメントに分かれているかを数えます（`bond_analysis`）。
`persistence` フレーム続けて分かれていた最初のフレームを、その分子の分解時刻とします。

In [ ]:
nonli = elec_idx[symbols[elec_idx] != "Li"]
nonli_mol = mol_of_atom[nonli]
n_fragments, intra_labels, census = bond_analysis(frames, nonli, mol_of_atom, len(mols), bond_scale=bond_scale)


def fragments_at(f, k):
    sel = nonli_mol == k
    return fragment_formula(symbols[nonli][sel], intra_labels[f][sel])


records = []
for k, mol in mols.iterrows():
    if mol["species"] == "Li":
        continue
    f_event = first_persistent(n_fragments[:, k] > 1, persistence)
    records.append({
        "label": mol["label"],
        "species": mol["species"],
        "closest_distance_t0_A": dist[0, mol["start"]:mol["end"]].min(),
        "decomposed": f_event is not None,
        "decomposition_time_ps": times[f_event] if f_event is not None else np.nan,
        "fragments_at_end": fragments_at(len(frames) - 1, k),
    })
events = pd.DataFrame(records)
events.to_csv(events_csv_file, index=False)

display(events.groupby("species", sort=False).agg(
    molecules=("label", "size"), decomposed=("decomposed", "sum"), first_event_ps=("decomposition_time_ps", "min")))
display(events[events["decomposed"]].sort_values("decomposition_time_ps"))

[EN]

## Reaction products

All non-Li atoms of the electrolyte are grouped into bonded species. Single atoms such as F are bound only to Li and
correspond to inorganic products such as LiF.

[JA]

## 反応生成物

電解液の Li 以外の原子を、結合でつながった化学種ごとに数えます。F のような単独の原子は Li とだけ結合しており、LiF のような無機の生成物に対応します。

In [ ]:
names = {hill(symbols[s:e]): sp for sp, s, e in zip(mols["species"], mols["start"], mols["end"]) if sp != "Li"}
products = pd.DataFrame({"t = 0": pd.Series(census[0]), f"t = {times[-1]:.1f} ps": pd.Series(census[-1])})
products = products.fillna(0).astype(int).sort_values(products.columns[-1], ascending=False)
products.index = [f"{s} ({names[s]})" if s in names else s for s in products.index]
products.to_csv(products_csv_file)
display(products.head(20))

In [ ]:
new_species = [s for s, n in census[-1].most_common() if s not in census[0]][:8]
fig, ax = plt.subplots(figsize=(7, 3.5))
for s in new_species:
    counts = pd.Series([c.get(s, 0) for c in census])
    ax.plot(times, counts.rolling(persistence, center=True, min_periods=1).median(), label=s)  # remove one-frame spikes
ax.set_xlabel("Time (ps)")
ax.set_ylabel("Number of species")
if new_species:
    ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
fig.savefig(products_plot_file, dpi=150)
plt.show()

[EN]

## Decomposition of selected molecules

For the earliest decomposed molecule of each species (or `closeup_molecules`), the table lists how the molecule is split
over time. A fragment is a group of bonded atoms: an intact molecule is one fragment, and EC that has lost CO is two
fragments, "C2H4O2 + CO".

With `replace_structure=True`, the viewer redraws the bonds in every frame, so broken bonds disappear. In the charge
plots, the red dashed lines mark the times when a bond breaks and the fragments change; a drop in charge at a dashed line
indicates that the molecule takes up electrons from Li as the bond breaks.

[JA]

## 分子の分解過程

分子種ごとに最も早く分解した分子（または `closeup_molecules` で指定した分子）について、分子の分かれ方を時刻ごとに一覧にします。ここでいうフラグメントは、結合でつながった原子のまとまりのことで、分解していない分子は 1 つのフラグメント、たとえば EC から CO が外れれば「C2H4O2 + CO」の 2 つのフラグメントになります。

ビューアは `replace_structure=True` でフレームごとに結合を描き直すので、切れた結合は表示されなくなります。電荷のグラフの赤い破線は、結合が切れて分子の分かれ方が変わった時刻です。破線のところで電荷が負の方向に下がっていれば、結合が切れるのと同時に、分子が Li から電子を受け取ったことを示しています。

In [ ]:
if closeup_molecules is None:
    earliest = events[events["decomposed"]].sort_values("decomposition_time_ps")
    closeup_molecules = earliest.groupby("species", sort=False).head(1)["label"].tolist()

closeups = {}
for label in closeup_molecules:
    k = int(np.where(mols["label"] == label)[0][0])
    states = [fragments_at(f, k) for f in range(len(frames))]
    rows, current = [], None
    for f in range(len(frames)):
        window = states[f:f + persistence]
        if states[f] != current and (f == 0 or (len(window) == persistence and len(set(window)) == 1)):
            current = states[f]  # record a change only if it lasts for `persistence` frames
            rows.append({"time_ps": times[f], "fragments": current,
                         "charge_e": q_mol[f, k] if charges is not None else np.nan})
    closeups[label] = (k, pd.DataFrame(rows))
    print(label)
    display(closeups[label][1])

In [ ]:
if charges is not None:
    for label, (k, timeline) in closeups.items():
        fig, ax = plt.subplots(figsize=(8, 3.2))
        ax.plot(times, q_mol[:, k])
        for t in timeline["time_ps"].iloc[1:]:
            ax.axvline(t, color="tab:red", lw=0.8, ls="--")
        ax.set_xlabel("Time (ps)")
        ax.set_ylabel("Charge (e)")
        ax.set_title(f"{label} (dashed: change of fragments)")
        fig.tight_layout()
        fig.savefig(str(closeup_plot_file).format(label=label), dpi=150)
        plt.show()

In [ ]:
for label, (k, timeline) in closeups.items():
    start, end = mols.loc[k, "start"], mols.loc[k, "end"]
    print(label)
    display(view_ngl([whole_molecule(f, start, end) for f in frames], representations=["ball+stick"],
                     replace_structure=True))

[EN]

## Composition near the Li surface

The SEI region contains the electrolyte-derived atoms within `sei_cutoff_A` of the outermost Li layer, including atoms
inside the slab, averaged over the last `composition_window_ps`. The enrichment factor is the ratio to the whole
electrolyte at t = 0. F and P include both decomposition products and intact PF<sub>6</sub><sup>−</sup> near the
positively charged surface.

[JA]

## 表面近くの組成

最外層の Li から `sei_cutoff_A` 以内（スラブの内側に入った原子を含む）にある電解液由来の原子を、計算の最後の `composition_window_ps` で平均します。
濃縮度は、t = 0 の電解液全体との比です。F と P には、分解生成物と、正に帯電した表面の近くにある分解していない PF<sub>6</sub><sup>−</sup> の両方が含まれます。

In [ ]:
window = np.where(times >= times[-1] - composition_window_ps)[0]
masses = np.array([atomic_masses[atomic_numbers[el]] for el in elements])
element_masks = np.array([(symbols == el) & (np.arange(n_atoms) >= n_slab) for el in elements])

counts = region_counts(dist[window], element_masks, sei_cutoff_A)
bulk = element_masks.sum(axis=1)
composition = pd.DataFrame({
    "SEI at%": to_percent(counts, masses, "atomic").mean(axis=0),
    "SEI at% (std)": to_percent(counts, masses, "atomic").std(axis=0),
    "SEI wt%": to_percent(counts, masses, "weight").mean(axis=0),
    "bulk at%": to_percent(bulk, masses, "atomic"),
    "bulk wt%": to_percent(bulk, masses, "weight"),
}, index=pd.Index(elements, name="element"))
composition["enrichment"] = composition["SEI at%"] / composition["bulk at%"]
composition.to_csv(composition_csv_file)
print(f"Within {sei_cutoff_A} A of the Li surface, {times[window[0]]:.1f}-{times[-1]:.1f} ps ({len(window)} frames)")
display(composition.round(2))

In [ ]:
sensitivity = pd.DataFrame(
    [to_percent(region_counts(dist[window], element_masks, c), masses, "atomic").mean(axis=0) for c in sensitivity_cutoffs],
    index=pd.Index(sensitivity_cutoffs, name="cutoff (A)"), columns=[f"{el} at%" for el in elements],
)
display(sensitivity.round(1))

In [ ]:
basis = "at%" if reference_composition is None or reference_composition["basis"] == "atomic" else "wt%"
sei_label = f"SEI region\n(within {sei_cutoff_A:g} A)"
table = pd.DataFrame({sei_label: composition[f"SEI {basis}"], "Whole electrolyte\n(t = 0)": composition[f"bulk {basis}"]})
if reference_composition is not None:
    table[reference_composition["label"]] = pd.Series(reference_composition["values"]).reindex(elements).fillna(0)

fig, ax = plt.subplots(figsize=(2.2 * len(table.columns) + 2, 4.5))
bottom = np.zeros(len(table.columns))
for el in elements:
    values = table.loc[el].to_numpy(dtype=float)
    ax.bar(table.columns, values, bottom=bottom, label=el)
    for x, (b, v) in enumerate(zip(bottom, values)):
        if v >= 3:  # label only segments large enough to read
            ax.text(x, b + v / 2, f"{v:.1f}", ha="center", va="center", color="white", fontsize=9)
    bottom += values
ax.set_ylabel(f"Composition ({basis})")
ax.set_ylim(0, 100)
ax.set_title(f"Average over {times[window[0]]:.1f}-{times[-1]:.1f} ps")
ax.legend(title="Element", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(composition_plot_file, dpi=150)
plt.show()

[EN]

## Summary

Pathways and enrichment trends are fairly reproducible between runs, but the amounts (decomposed molecules, transferred
charge) depend strongly on the initial structure. For quantitative discussion, use a larger system (a larger in-plane cell
with more molecules) or average several runs with different initial structures.

[JA]

## まとめ

分解経路や濃縮の傾向は計算ごとに比較的よく再現しますが、量（分解した分子の数、移った電荷）は初期構造に強く依存します。定量的に議論するには、系を大きくする（面内のサイズを広げて分子の数を増やす）か、初期構造を変えた計算を複数回行って平均してください。

[ALL]

Copyright 2026 Matlantis Corp. as contributors to Matlantis contrib project (supported by Tata Consultancy services Ltd.)